# Qwen3 Dream Continuation Evaluation Notebook — Four Models + Semantic Similarity

This notebook evaluates four variants of the dream-continuation system:

1. **Base Qwen3 without metadata**
2. **Base Qwen3 with metadata**
3. **Fine-tuned Qwen3 without metadata**
4. **Fine-tuned Qwen3 with metadata**

It computes the heavy metrics here:

- **NLL / perplexity** on the true continuation only
- **Generated continuation** for each model variant
- **Embedding cosine similarity** between the true continuation and the generated continuation
- Summary and pairwise comparison CSV files

No plots are produced in this notebook. The plots are produced later in the VS Code interactive notebook by loading the exported CSV files.


## 1. Install packages

Run this first in Colab with GPU enabled: **Runtime → Change runtime type → GPU**.

In [1]:
%pip install -U "transformers>=4.51.0" "accelerate" "datasets" "peft" "bitsandbytes" "pandas" "tqdm" "safetensors" "scikit-learn" "ipywidgets" "sentence-transformers"


## 2. Imports, central file inputs, and configuration

Edit this cell first. In Colab, you can leave the file paths as `None`.

The notebook now opens **two separate upload dialogs**:

1. one for the dream CSV dataset;
2. one for the zip exported by the fine-tuning notebook, usually `qwen3_dream_final_lora_adapters.zip`.


In [2]:
import os
import gc
import math
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

SEED = 42
set_seed(SEED)
np.random.seed(SEED)

# ============================================================
# 0. CENTRAL INPUT FILES — EDIT ONLY THIS PART WHEN NEEDED
# ============================================================
# In Colab, leave these as None and upload all files when this cell asks.
# Outside Colab, set them to explicit local paths, for example:
# CSV_PATH = "dream_clean_emotion_topic_metadata.csv"
# FINETUNED_ADAPTERS_ZIP_PATH = "qwen3_dream_metadata_lora_adapters.zip"
CSV_PATH = None
FINETUNED_ADAPTERS_ZIP_PATH = None

# Base model used in your fine-tuning notebook.
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"

# The fine-tuning notebook creates this zip by default:
# qwen3_dream_metadata_lora_adapters.zip
# When extracted, the adapter folders may appear either under:
#   qwen3_dream_cv_final_outputs/final_adapters/...
# or directly under:
#   final_adapters/...
FINETUNED_ROOT = Path("qwen3_dream_cv_final_outputs")
ADAPTER_A_NAME = "final_A_no_metadata"
ADAPTER_B_NAME = "final_B_with_metadata"

# If True, the first configuration cell asks for all missing files once.
UPLOAD_ALL_INPUTS_AT_START = True
AUTO_EXTRACT_ADAPTER_ZIP = True

# Dataset columns. The notebook also tries alternatives for the text column.
TEXT_COL = "dream_text"

# All metadata fields inserted in the prompt for the metadata-conditioned variants.
METADATA_COLS = [
    "dominant_macro_emotion",
    "second_dominant_macro_emotion",
    "dream_topic",
    "sex",
    "age",
]

TEXT_COL_CANDIDATES = ["dream_text", "text_dream", "dream", "text", "report"]

# Same split logic as the fine-tuning notebook.
SPLIT_RATIO = 0.70
MIN_WORDS = 30
MAX_WORDS = 350

# Token limits. Keep these aligned with the fine-tuning notebook.
MAX_LENGTH = 768
MAX_TARGET_LENGTH = 256

# Evaluation size. Start small for speed; set to None for the full cleaned dataset.
# Example: 300 gives a useful quick comparison; full evaluation can take a long time.
N_EVAL_SAMPLES = 300

# Use 4-bit quantization on GPU. This is recommended for Colab.
USE_4BIT = True

# Output files created by this evaluation notebook.
RESULTS_CSV = "qwen3_dream_four_model_metadata_evaluation_results.csv"
SUMMARY_CSV = "qwen3_dream_four_model_metadata_summary.csv"
PAIRWISE_CSV = "qwen3_dream_four_model_metadata_pairwise_comparisons.csv"
SEMANTIC_CSV = "qwen3_dream_four_model_metadata_semantic_similarity.csv"

# Semantic generation and embedding settings.
# The generation is deterministic by default, so that the evaluation is reproducible.
GENERATE_CONTINUATIONS_FOR_SEMANTIC_EVAL = True
SEMANTIC_GENERATION_MAX_NEW_TOKENS = 160
SEMANTIC_GENERATION_MIN_NEW_TOKENS = 40
SEMANTIC_GENERATION_TARGET_TOKEN_MULTIPLIER = 1.10
SEMANTIC_GENERATION_DO_SAMPLE = False
SEMANTIC_GENERATION_TEMPERATURE = 0.80
SEMANTIC_GENERATION_TOP_P = 0.90
SEMANTIC_GENERATION_REPETITION_PENALTY = 1.08

# Embedding model used to compare real endings and generated endings.
# BGE is a sentence embedding model; cosine similarity close to 1 means semantically similar texts.
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_BATCH_SIZE = 32

# ============================================================
# Startup upload/extract helpers
# ============================================================
def _path_exists(path_value):
    return path_value is not None and Path(path_value).exists()


def _detect_local_file(extensions, prefer_keywords=None):
    """Detect an already uploaded local file, useful when rerunning the cell in Colab."""
    prefer_keywords = prefer_keywords or []
    candidates = []
    for ext in extensions:
        candidates.extend(Path(".").glob(f"*{ext}"))
        candidates.extend(Path(".").glob(f"*{ext.upper()}"))
    candidates = sorted(set(candidates))
    if not candidates:
        return None
    preferred = [
        path for path in candidates
        if any(keyword.lower() in path.name.lower() for keyword in prefer_keywords)
    ]
    return str(preferred[0] if preferred else candidates[0])


def _choose_uploaded_file(uploaded_names, extensions, prefer_keywords=None):
    """Choose the most relevant uploaded file with the requested extension."""
    prefer_keywords = prefer_keywords or []
    matching = [
        name for name in uploaded_names
        if any(name.lower().endswith(ext.lower()) for ext in extensions)
    ]
    if not matching:
        return None
    preferred = [
        name for name in matching
        if any(keyword.lower() in name.lower() for keyword in prefer_keywords)
    ]
    return preferred[0] if preferred else matching[0]


def _quick_adapter_pair_exists():
    """Fast check before the full recursive adapter search is defined later."""
    candidate_pairs = [
        (FINETUNED_ROOT / "final_adapters" / ADAPTER_A_NAME, FINETUNED_ROOT / "final_adapters" / ADAPTER_B_NAME),
        (Path("final_adapters") / ADAPTER_A_NAME, Path("final_adapters") / ADAPTER_B_NAME),
        (Path(ADAPTER_A_NAME), Path(ADAPTER_B_NAME)),
    ]
    for adapter_a, adapter_b in candidate_pairs:
        if (adapter_a / "adapter_config.json").exists() and (adapter_b / "adapter_config.json").exists():
            return True
    return False


def _extract_adapter_zip_if_available():
    if not AUTO_EXTRACT_ADAPTER_ZIP:
        return
    if not _path_exists(FINETUNED_ADAPTERS_ZIP_PATH):
        return
    if _quick_adapter_pair_exists():
        print("Fine-tuned adapter folders already found. No extraction needed.")
        return
    print("Extracting adapter zip:", FINETUNED_ADAPTERS_ZIP_PATH)
    shutil.unpack_archive(str(FINETUNED_ADAPTERS_ZIP_PATH), ".")


def upload_all_inputs_at_start():
    """
    Upload the required files using TWO SEPARATE Colab upload dialogs:
    1) one dialog for the dream CSV;
    2) one dialog for the LoRA adapter zip.

    This avoids the problem where Colab lets you select/upload only one file.
    """
    global CSV_PATH, FINETUNED_ADAPTERS_ZIP_PATH

    if not IN_COLAB or not UPLOAD_ALL_INPUTS_AT_START:
        return {}

    uploaded_all = {}

    # --------------------------------------------------------
    # Step 1: resolve or upload the CSV dataset
    # --------------------------------------------------------
    if not _path_exists(CSV_PATH):
        detected_csv = _detect_local_file([".csv"], prefer_keywords=["dream", "goemotions"])
        if detected_csv is not None:
            CSV_PATH = detected_csv
            print("Detected existing CSV file:", CSV_PATH)

    if not _path_exists(CSV_PATH):
        print("STEP 1/2 - Upload the DREAM DATASET CSV now.")
        print("Example: dreams_goemotions_10macro_full.csv")
        uploaded_csv = files.upload()
        uploaded_all.update(uploaded_csv)
        chosen_csv = _choose_uploaded_file(
            list(uploaded_csv.keys()),
            extensions=[".csv"],
            prefer_keywords=["dream", "goemotions"]
        )
        if chosen_csv is None:
            raise FileNotFoundError("No CSV file was uploaded in STEP 1/2.")
        CSV_PATH = chosen_csv

    # --------------------------------------------------------
    # Step 2: resolve or upload the fine-tuned adapter zip
    # --------------------------------------------------------
    if not _quick_adapter_pair_exists() and not _path_exists(FINETUNED_ADAPTERS_ZIP_PATH):
        detected_zip = _detect_local_file([".zip"], prefer_keywords=["adapter", "qwen3_dream", "lora"])
        if detected_zip is not None:
            FINETUNED_ADAPTERS_ZIP_PATH = detected_zip
            print("Detected existing adapter zip:", FINETUNED_ADAPTERS_ZIP_PATH)

    if not _quick_adapter_pair_exists() and not _path_exists(FINETUNED_ADAPTERS_ZIP_PATH):
        print("STEP 2/2 - Upload the FINE-TUNED LoRA ADAPTER ZIP now.")
        print("Usually: qwen3_dream_final_lora_adapters.zip")
        print("This zip is needed for the two fine-tuned models.")
        uploaded_zip = files.upload()
        uploaded_all.update(uploaded_zip)
        chosen_zip = _choose_uploaded_file(
            list(uploaded_zip.keys()),
            extensions=[".zip"],
            prefer_keywords=["adapter", "qwen3_dream", "lora"]
        )
        if chosen_zip is None:
            print("WARNING: no zip file uploaded in STEP 2/2.")
            print("Base models can still run, but fine-tuned models need the adapter zip or extracted adapter folders.")
        else:
            FINETUNED_ADAPTERS_ZIP_PATH = chosen_zip

    _extract_adapter_zip_if_available()
    return uploaded_all


UPLOADED_AT_START = upload_all_inputs_at_start()

print("Running in Colab:", IN_COLAB)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("CSV_PATH:", CSV_PATH)
print("FINETUNED_ADAPTERS_ZIP_PATH:", FINETUNED_ADAPTERS_ZIP_PATH)
print("Fine-tuned adapter folders found:", _quick_adapter_pair_exists())


STEP 1/2 - Upload the DREAM DATASET CSV now.
Example: dreams_goemotions_10macro_full.csv


Saving dream_clean_emotion_topic_metadata.csv to dream_clean_emotion_topic_metadata.csv
STEP 2/2 - Upload the FINE-TUNED LoRA ADAPTER ZIP now.
Usually: qwen3_dream_final_lora_adapters.zip
This zip is needed for the two fine-tuned models.


Saving qwen3_dream_metadata_lora_adapters.zip to qwen3_dream_metadata_lora_adapters.zip
Extracting adapter zip: qwen3_dream_metadata_lora_adapters.zip
Running in Colab: True
CUDA available: True
GPU: Tesla T4
CSV_PATH: dream_clean_emotion_topic_metadata.csv
FINETUNED_ADAPTERS_ZIP_PATH: qwen3_dream_metadata_lora_adapters.zip
Fine-tuned adapter folders found: True


## 3. Load the uploaded/input dream dataset

This cell now reuses `CSV_PATH` from the first configuration cell. It only asks for a CSV again if no usable CSV was supplied at the start.


In [3]:
def resolve_csv_path(csv_path=None):
    """Return a valid CSV path. The preferred path is supplied in the first configuration cell."""
    if csv_path is not None and Path(csv_path).exists():
        return str(csv_path)

    # If the startup upload put a CSV in the working directory, reuse it.
    local_csvs = sorted(Path(".").glob("*.csv"))
    if local_csvs:
        dream_csvs = [path for path in local_csvs if "dream" in path.name.lower()]
        chosen = dream_csvs[0] if dream_csvs else local_csvs[0]
        print("Using detected CSV file:", chosen)
        return str(chosen)

    if IN_COLAB:
        print("No CSV found. Upload your dream dataset CSV now.")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded.")
        return csv_files[0]

    raise FileNotFoundError(
        "CSV_PATH is None and no CSV was found. Set CSV_PATH to your local CSV path in the first configuration cell."
    )

CSV_PATH = resolve_csv_path(CSV_PATH)
df_raw = pd.read_csv(CSV_PATH)

print("CSV path:", CSV_PATH)
print("Original shape:", df_raw.shape)
print("Columns:")
print(df_raw.columns.tolist())
display(df_raw.head())


CSV path: dream_clean_emotion_topic_metadata.csv
Original shape: (21000, 6)
Columns:
['dream_text', 'dominant_macro_emotion', 'second_dominant_macro_emotion', 'dream_topic', 'sex', 'age']


,dream_text,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,"The one at the Meads's house, where it's bigge...",admiration_approval,joy_amusement,topic_02_wheelchair_bonnie_yarn_dovre,female,adult
1,I'm at a family reunion in a large fine house ...,sadness_grief,disappointment_remorse,topic_15_ski_pilot_tornado_clouds,female,adult
2,I watch a plane fly past and shortly realize i...,embarrassment_confusion,disgust_disapproval,topic_15_ski_pilot_tornado_clouds,female,adult
3,Me pulling the green leaves and berries off so...,embarrassment_confusion,admiration_approval,topic_17_kitten_kittens_patio_plant,female,adult
4,I'm in a room that reminds me of (but definite...,embarrassment_confusion,curiosity_surprise,topic_02_wheelchair_bonnie_yarn_dovre,female,adult


## 4. Clean the dataset with the same logic as fine-tuning

This removes missing text/metadata rows and keeps dreams within the selected word-count range.

In [4]:
def find_column(dataframe, preferred_col, candidates, role):
    if preferred_col in dataframe.columns:
        return preferred_col
    for col in candidates:
        if col in dataframe.columns:
            return col
    raise ValueError(
        f"No {role} column found. Tried {preferred_col} and {candidates}. "
        f"Available columns: {dataframe.columns.tolist()}"
    )


def validate_metadata_columns(dataframe, metadata_cols):
    missing = [col for col in metadata_cols if col not in dataframe.columns]
    if missing:
        raise ValueError(
            f"Missing metadata columns: {missing}. "
            f"Available columns: {dataframe.columns.tolist()}"
        )


TEXT_COL = find_column(df_raw, TEXT_COL, TEXT_COL_CANDIDATES, "text")
validate_metadata_columns(df_raw, METADATA_COLS)

# Clean text and metadata.
df_clean = df_raw.copy()
required_cols = [TEXT_COL] + METADATA_COLS
df_clean = df_clean.dropna(subset=required_cols).copy()

df_clean[TEXT_COL] = df_clean[TEXT_COL].astype(str).str.strip()
for col in METADATA_COLS:
    df_clean[col] = df_clean[col].astype(str).str.strip()

invalid_values = {"", "nan", "none", "null"}
df_clean = df_clean[~df_clean[TEXT_COL].str.lower().isin(invalid_values)].copy()
for col in METADATA_COLS:
    df_clean = df_clean[~df_clean[col].str.lower().isin(invalid_values)].copy()

# Recompute n_words to avoid relying on stale columns.
df_clean["n_words"] = df_clean[TEXT_COL].astype(str).str.split().str.len()
df_clean = df_clean[(df_clean["n_words"] >= MIN_WORDS) & (df_clean["n_words"] <= MAX_WORDS)].copy()
df_clean = df_clean.reset_index(drop=True)

print("Text column used:", TEXT_COL)
print("Metadata columns used:", METADATA_COLS)
print("Rows after cleaning:", len(df_clean))
print("Removed rows:", len(df_raw) - len(df_clean))
print("Word-count summary:")
display(df_clean["n_words"].describe())
print("Metadata distributions:")
for col in METADATA_COLS:
    print(f"\n{col} distribution:")
    display(df_clean[col].value_counts().to_frame("count"))
display(df_clean[[TEXT_COL] + METADATA_COLS + ["n_words"]].head())

Text column used: dream_text
Metadata columns used: ['dominant_macro_emotion', 'second_dominant_macro_emotion', 'dream_topic', 'sex', 'age']
Rows after cleaning: 18093
Removed rows: 2907
Word-count summary:


count    18093.000000
mean       147.147958
std         76.368712
min         50.000000
25%         84.000000
50%        128.000000
75%        195.000000
max        350.000000
Name: n_words, dtype: float64

Metadata distributions:

dominant_macro_emotion distribution:


,count
dominant_macro_emotion,
admiration_approval,5308
embarrassment_confusion,1915
joy_amusement,1773
anger_frustration,1647
fear_anxiety,1585
disappointment_remorse,1493
sadness_grief,1328
curiosity_surprise,1100
optimism_desire,632



second_dominant_macro_emotion distribution:


...


sex distribution:


,count
sex,
female,12506
male,5534
mixed_or_group,53



age distribution:


,count
age,
unknown,4280
longitudinal_multiple_ages,4173
middle_aged,4111
young_adult,2573
adolescent,1129
mixed_or_group,948
adult,404
child,319
older_adult,156


,dream_text,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words
0,"The one at the Meads's house, where it's bigge...",admiration_approval,joy_amusement,topic_02_wheelchair_bonnie_yarn_dovre,female,adult,167
1,I'm at a family reunion in a large fine house ...,sadness_grief,disappointment_remorse,topic_15_ski_pilot_tornado_clouds,female,adult,258
2,I watch a plane fly past and shortly realize i...,embarrassment_confusion,disgust_disapproval,topic_15_ski_pilot_tornado_clouds,female,adult,330
3,Living next door to Loretta in an apartment - ...,embarrassment_confusion,admiration_approval,topic_02_wheelchair_bonnie_yarn_dovre,female,adult,227
4,Kidnapped - I'm on my way somewhere else (by c...,embarrassment_confusion,admiration_approval,topic_03_aliens_enemy_guns_shooting,female,adult,286


## 5. Split each dream into beginning and continuation

The model sees only the beginning as context. The loss is computed only on the hidden continuation.

In [5]:
def split_dream_report(text, ratio=SPLIT_RATIO):
    words = str(text).split()
    if len(words) < 2:
        return "", ""

    split_idx = int(len(words) * ratio)
    split_idx = max(1, split_idx)
    split_idx = min(split_idx, len(words) - 1)

    prefix = " ".join(words[:split_idx]).strip()
    continuation = " ".join(words[split_idx:]).strip()
    return prefix, continuation

prefixes = []
continuations = []

for dream in df_clean[TEXT_COL].tolist():
    prefix, continuation = split_dream_report(dream)
    prefixes.append(prefix)
    continuations.append(continuation)

df_clean["prefix"] = prefixes
df_clean["continuation"] = continuations

df_clean = df_clean[(df_clean["prefix"].str.len() > 0) & (df_clean["continuation"].str.len() > 0)].copy()
df_clean = df_clean.reset_index(drop=True)

# Sample for evaluation speed.
if N_EVAL_SAMPLES is not None:
    n_eval = min(int(N_EVAL_SAMPLES), len(df_clean))
    df_eval = df_clean.sample(n=n_eval, random_state=SEED).reset_index(drop=True)
else:
    df_eval = df_clean.reset_index(drop=True)

# Stable row id for pairwise comparisons across all four variants.
df_eval["row_id"] = np.arange(len(df_eval))

print("Rows available after split:", len(df_clean))
print("Rows selected for evaluation:", len(df_eval))
display(df_eval[["row_id", "prefix", "continuation"] + METADATA_COLS + ["n_words"]].head())

Rows available after split: 18093
Rows selected for evaluation: 300


,row_id,prefix,continuation,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words
0,0,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown,191
1,1,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult,53
2,2,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group,68
3,3,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...",joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages,70
4,4,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group,126


## 6. Prompt templates

Use the same prompt style as the fine-tuning notebook: no-metadata variants receive only `Dream beginning`; with-metadata variants receive a `Dream metadata` block plus the same beginning.

In [6]:
METADATA_LABELS = {
    "dominant_macro_emotion": "dominant_macro_emotion",
    "second_dominant_macro_emotion": "second_dominant_macro_emotion",
    "dream_topic": "dream_topic",
    "sex": "sex",
    "age": "age",
}


def clean_metadata_value(value):
    value = str(value).strip()
    if value == "" or value.lower() in {"nan", "none", "null"}:
        return "unknown"
    return value


def extract_metadata_from_row(row):
    return {col: clean_metadata_value(row[col]) for col in METADATA_COLS}


def format_metadata_block(metadata):
    lines = ["Dream metadata:"]
    for col in METADATA_COLS:
        label = METADATA_LABELS.get(col, col)
        value = clean_metadata_value(metadata.get(col, "unknown"))
        lines.append(f"- {label}: {value}")
    return "\n".join(lines)


def make_prompt(prefix, metadata=None):
    if metadata is None:
        return (
            "Dream beginning:\n"
            f"{prefix}\n\n"
            "Dream continuation:\n"
        )

    return (
        f"{format_metadata_block(metadata)}\n\n"
        "Dream beginning:\n"
        f"{prefix}\n\n"
        "Dream continuation:\n"
    )


example_row = df_eval.iloc[0]
print("Prompt without metadata:\n")
print(make_prompt(example_row["prefix"], metadata=None)[:1000])
print("\nPrompt with all metadata:\n")
print(make_prompt(example_row["prefix"], metadata=extract_metadata_from_row(example_row))[:1200])

Prompt without metadata:

Dream beginning:
Bigger Burger For $1.09 I'm just eaten some food from the Greasepit. I hop in my truck. My friend Kevin Simpson is driving. His truck is fixed, and the bumper is on my truck. I'm happy to see him! I ask him if he's hungry: if he wants to eat, and where. I tell him it's my treat: I'll pay. He says the Greasepit! I tell him they've lowered the price of a Bigger Burger from $1.50 back to $1.09, the previous price. I'm on a bicycle at the intersection of Main Street and Last Avenue in Oak Valley near Tom Thumb. A tall blond man is next to me. I swing my arm to hit him in the face with the back of my hand. He catches my hand in his mouth,

Dream continuation:


Prompt with all metadata:

Dream metadata:
- dominant_macro_emotion: joy_amusement
- second_dominant_macro_emotion: admiration_approval
- dream_topic: topic_04_cookies_bread_grocery_store_pan
- sex: male
- age: unknown

Dream beginning:
Bigger Burger For $1.09 I'm just eaten some food from t

## 7. Locate, extract, and validate the fine-tuned adapter folders

The file created by the fine-tuning notebook is a **zip containing LoRA adapter folders**, not a full standalone model. This cell extracts the zip if needed and checks that both expected adapters are available.


In [7]:
def find_adapter_dir(adapter_name, verbose=True):
    """Find a LoRA adapter directory by checking common layouts and then searching recursively."""
    direct_candidates = [
        FINETUNED_ROOT / "final_adapters" / adapter_name,
        Path("final_adapters") / adapter_name,
        Path(adapter_name),
    ]

    for candidate in direct_candidates:
        if (candidate / "adapter_config.json").exists():
            if verbose:
                print(f"Found {adapter_name} at: {candidate}")
            return candidate

    matches = []
    for config_path in Path(".").rglob("adapter_config.json"):
        if config_path.parent.name == adapter_name:
            matches.append(config_path.parent)

    if matches:
        if verbose:
            print(f"Found {adapter_name} at: {matches[0]}")
        return matches[0]

    return None


def extract_adapter_zip_if_needed():
    if _quick_adapter_pair_exists():
        return

    if FINETUNED_ADAPTERS_ZIP_PATH is not None and Path(FINETUNED_ADAPTERS_ZIP_PATH).exists():
        print("Extracting adapter zip:", FINETUNED_ADAPTERS_ZIP_PATH)
        shutil.unpack_archive(str(FINETUNED_ADAPTERS_ZIP_PATH), ".")
        return

    # Fallback only if the first cell did not receive the zip.
    if IN_COLAB:
        print("Adapter folders were not found. Upload qwen3_dream_final_lora_adapters.zip now.")
        uploaded = files.upload()
        zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
        for zip_file in zip_files:
            print("Extracting:", zip_file)
            shutil.unpack_archive(zip_file, ".")


def validate_lora_adapter(adapter_dir, expected_base_model=MODEL_NAME):
    """Light compatibility check for a PEFT/LoRA adapter folder."""
    adapter_dir = Path(adapter_dir)
    config_path = adapter_dir / "adapter_config.json"
    model_files = [adapter_dir / "adapter_model.safetensors", adapter_dir / "adapter_model.bin"]

    report = {
        "adapter_dir": str(adapter_dir),
        "has_adapter_config": config_path.exists(),
        "has_adapter_weights": any(path.exists() for path in model_files),
        "has_tokenizer_config": (adapter_dir / "tokenizer_config.json").exists(),
        "base_model_in_config": None,
        "base_model_matches": None,
    }

    if config_path.exists():
        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        base_model_in_config = cfg.get("base_model_name_or_path")
        report["base_model_in_config"] = base_model_in_config
        report["base_model_matches"] = (
            base_model_in_config is None
            or str(base_model_in_config).rstrip("/") == str(expected_base_model).rstrip("/")
        )

    report["usable_by_this_notebook"] = bool(
        report["has_adapter_config"]
        and report["has_adapter_weights"]
        and (report["base_model_matches"] is not False)
    )
    return report


def maybe_upload_and_extract_adapters():
    adapter_a = find_adapter_dir(ADAPTER_A_NAME, verbose=False)
    adapter_b = find_adapter_dir(ADAPTER_B_NAME, verbose=False)

    if adapter_a is None or adapter_b is None:
        extract_adapter_zip_if_needed()
        adapter_a = find_adapter_dir(ADAPTER_A_NAME, verbose=True)
        adapter_b = find_adapter_dir(ADAPTER_B_NAME, verbose=True)
    else:
        print("Both adapter folders are already available.")

    if adapter_a is None or adapter_b is None:
        raise FileNotFoundError(
            "Could not find both final LoRA adapters. Expected folders named "
            f"{ADAPTER_A_NAME} and {ADAPTER_B_NAME}, each containing adapter_config.json. "
            "Upload or unzip qwen3_dream_final_lora_adapters.zip."
        )

    return adapter_a, adapter_b


ADAPTER_A_DIR, ADAPTER_B_DIR = maybe_upload_and_extract_adapters()
print("Adapter A:", ADAPTER_A_DIR)
print("Adapter B:", ADAPTER_B_DIR)

adapter_reports = pd.DataFrame([
    validate_lora_adapter(ADAPTER_A_DIR),
    validate_lora_adapter(ADAPTER_B_DIR),
])
print("Adapter compatibility check:")
display(adapter_reports)

if not adapter_reports["usable_by_this_notebook"].all():
    raise ValueError(
        "At least one adapter folder is incomplete or incompatible. Check the zip exported by the fine-tuning notebook."
    )
else:
    print("Compatibility result: OK. The fine-tuning zip can be imported by this notebook as PEFT/LoRA adapters.")


Both adapter folders are already available.
Adapter A: final_adapters/final_A_no_metadata
Adapter B: final_adapters/final_B_with_metadata
Adapter compatibility check:


,adapter_dir,has_adapter_config,has_adapter_weights,has_tokenizer_config,base_model_in_config,base_model_matches,usable_by_this_notebook
0,final_adapters/final_A_no_metadata,True,True,True,Qwen/Qwen3-1.7B-Base,True,True
1,final_adapters/final_B_with_metadata,True,True,True,Qwen/Qwen3-1.7B-Base,True,True


Compatibility result: OK. The fine-tuning zip can be imported by this notebook as PEFT/LoRA adapters.


## 8. Model loading utilities

The base model is loaded once for the two non-fine-tuned evaluations. Then the two LoRA adapters are loaded separately.

In [8]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def get_compute_dtype():
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32


def make_quantization_config():
    if USE_4BIT and torch.cuda.is_available():
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=get_compute_dtype(),
        )
    return None


def load_tokenizer(tokenizer_source=None):
    source = tokenizer_source if tokenizer_source is not None else MODEL_NAME
    tokenizer = AutoTokenizer.from_pretrained(str(source), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer


def load_base_model_and_tokenizer():
    clear_memory()
    tokenizer = load_tokenizer(MODEL_NAME)
    quantization_config = make_quantization_config()

    model_kwargs = dict(
        trust_remote_code=True,
        torch_dtype=get_compute_dtype(),
    )

    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config
        model_kwargs["device_map"] = "auto"
    else:
        model_kwargs["device_map"] = "auto" if torch.cuda.is_available() else None

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    model.eval()
    return model, tokenizer


def load_lora_model_and_tokenizer(adapter_dir):
    clear_memory()
    adapter_dir = Path(adapter_dir)

    # The fine-tuning notebook saves tokenizer files inside the adapter folder.
    tokenizer_source = adapter_dir if (adapter_dir / "tokenizer_config.json").exists() else MODEL_NAME
    tokenizer = load_tokenizer(tokenizer_source)

    quantization_config = make_quantization_config()
    model_kwargs = dict(
        trust_remote_code=True,
        torch_dtype=get_compute_dtype(),
    )

    if quantization_config is not None:
        model_kwargs["quantization_config"] = quantization_config
        model_kwargs["device_map"] = "auto"
    else:
        model_kwargs["device_map"] = "auto" if torch.cuda.is_available() else None

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
    model = PeftModel.from_pretrained(base_model, str(adapter_dir))
    model.eval()
    return model, tokenizer


def get_model_device(model):
    return next(model.parameters()).device

## 9. NLL and perplexity on the continuation only

The prompt is used as context but is masked with `-100`, so it does not contribute to the loss. Only the continuation contributes to NLL/perplexity.

In [9]:
def safe_exp(x):
    try:
        return float(math.exp(float(x)))
    except OverflowError:
        return float("inf")


def encode_prompt_and_target(tokenizer, prompt, target):
    """
    Tokenize prompt + target using the same truncation logic as the fine-tuning notebook.
    The target is always kept as much as possible; if needed, the beginning of the prompt is truncated.
    """
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target + tokenizer.eos_token,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = target_ids[:MAX_TARGET_LENGTH]
    available_prompt_len = MAX_LENGTH - len(target_ids)

    if available_prompt_len < 0:
        target_ids = target_ids[:MAX_LENGTH]
        prompt_ids = []
    else:
        prompt_ids = prompt_ids[-available_prompt_len:]

    input_ids = prompt_ids + target_ids
    attention_mask = [1] * len(input_ids)
    labels = [-100] * len(prompt_ids) + target_ids.copy()

    return input_ids, attention_mask, labels


def compute_target_nll_ppl(model, tokenizer, prompt, target):
    """
    Compute NLL and perplexity only on the target continuation.
    Returns average NLL per evaluated target token and total NLL.
    """
    input_ids, attention_mask, labels = encode_prompt_and_target(tokenizer, prompt, target)

    if len(input_ids) < 2:
        raise ValueError("Sequence is too short after tokenization.")

    device = get_model_device(model)
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)
    attention_mask = torch.tensor([attention_mask], dtype=torch.long, device=device)
    labels = torch.tensor([labels], dtype=torch.long, device=device)

    # Hugging Face causal LM loss predicts labels[:, 1:] from logits[:, :-1].
    # This count matches the actual number of tokens included in the internal loss.
    effective_target_tokens = int((labels[:, 1:] != -100).sum().item())
    if effective_target_tokens == 0:
        raise ValueError("No target tokens were left for evaluation after masking/truncation.")

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

    avg_nll = float(outputs.loss.item())
    total_nll = avg_nll * effective_target_tokens
    ppl = safe_exp(avg_nll)

    return {
        "avg_nll": avg_nll,
        "total_nll": total_nll,
        "ppl": ppl,
        "target_tokens": effective_target_tokens,
    }


def choose_semantic_generation_length(target_tokens):
    """
    Choose a generation length close to the true continuation length, but capped for speed.
    This makes the semantic comparison more balanced across examples.
    """
    try:
        target_tokens = int(target_tokens)
    except Exception:
        target_tokens = SEMANTIC_GENERATION_MIN_NEW_TOKENS

    proposed = int(round(target_tokens * SEMANTIC_GENERATION_TARGET_TOKEN_MULTIPLIER))
    proposed = max(SEMANTIC_GENERATION_MIN_NEW_TOKENS, proposed)
    proposed = min(SEMANTIC_GENERATION_MAX_NEW_TOKENS, proposed)
    return int(proposed)


def generate_continuation_for_semantic_eval(model, tokenizer, prompt, target_tokens=None):
    """
    Generate one deterministic continuation for semantic evaluation.

    We use greedy decoding by default (`do_sample=False`) so that the same notebook run
    gives reproducible generated endings. This generated text is then compared with the
    true continuation using embedding cosine similarity.
    """
    if not GENERATE_CONTINUATIONS_FOR_SEMANTIC_EVAL:
        return "", 0

    max_new_tokens = choose_semantic_generation_length(target_tokens)
    prompt_max_length = max(64, MAX_LENGTH - max_new_tokens)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=prompt_max_length,
    )

    device = get_model_device(model)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    generate_kwargs = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": bool(SEMANTIC_GENERATION_DO_SAMPLE),
        "repetition_penalty": float(SEMANTIC_GENERATION_REPETITION_PENALTY),
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if SEMANTIC_GENERATION_DO_SAMPLE:
        generate_kwargs["temperature"] = float(SEMANTIC_GENERATION_TEMPERATURE)
        generate_kwargs["top_p"] = float(SEMANTIC_GENERATION_TOP_P)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generate_kwargs,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    continuation = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return continuation, max_new_tokens


## 10. Evaluation function for one variant

Each variant is evaluated on exactly the same rows, so pairwise comparisons are fair.

### Progress-bar evaluation update

The evaluation function below now shows a live progress bar for each model variant. The postfix indicates whether the current example is computing **NLL** or doing **GEN** generation, plus the last/average seconds per dream and error counters.


In [10]:
# Evaluation function with detailed progress bars.
# It shows whether each example is doing NLL/perplexity or generation,
# plus elapsed time, average seconds per example, and error counters.
import time
import warnings

try:
    from tqdm.notebook import tqdm as progress_tqdm
except Exception:
    from tqdm.auto import tqdm as progress_tqdm

# Hide harmless bitsandbytes/PyTorch FutureWarnings that can make the output look blocked.
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r".*_check_is_size.*",
)


def evaluate_variant(model, tokenizer, dataframe, model_variant, use_metadata):
    records = []
    n_errors = 0
    n_generation_errors = 0
    start_time = time.time()

    progress_bar = progress_tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"Evaluating {model_variant}",
        unit="dream",
        dynamic_ncols=True,
        leave=True,
    )

    for i, (_, row) in enumerate(progress_bar, start=1):
        example_start_time = time.time()

        metadata = extract_metadata_from_row(row) if use_metadata else None
        prompt = make_prompt(row["prefix"], metadata=metadata)
        target_for_loss = " " + str(row["continuation"]).strip()
        target_clean = str(row["continuation"]).strip()

        progress_bar.set_postfix(
            step="NLL",
            errors=n_errors,
            gen_errors=n_generation_errors,
            refresh=True,
        )

        try:
            metrics = compute_target_nll_ppl(model, tokenizer, prompt, target_for_loss)
            error = ""
        except Exception as exc:
            metrics = {
                "avg_nll": np.nan,
                "total_nll": np.nan,
                "ppl": np.nan,
                "target_tokens": np.nan,
            }
            error = repr(exc)
            n_errors += 1

        generated_continuation = ""
        generation_max_new_tokens = np.nan
        generation_error = ""

        if error == "" and GENERATE_CONTINUATIONS_FOR_SEMANTIC_EVAL:
            progress_bar.set_postfix(
                step="GEN",
                errors=n_errors,
                gen_errors=n_generation_errors,
                refresh=True,
            )

            try:
                generated_continuation, generation_max_new_tokens = generate_continuation_for_semantic_eval(
                    model=model,
                    tokenizer=tokenizer,
                    prompt=prompt,
                    target_tokens=metrics["target_tokens"],
                )
            except Exception as exc:
                generated_continuation = ""
                generation_error = repr(exc)
                n_generation_errors += 1

        record = {
            "row_id": int(row["row_id"]),
            "model_variant": model_variant,
            "use_metadata": bool(use_metadata),
            "dream_id": row.get("dream_id", row["row_id"]),
            "dreamer": row.get("dreamer", ""),
            "metadata_text": format_metadata_block(extract_metadata_from_row(row)),
            "n_words": int(row["n_words"]),
            "prefix": str(row["prefix"]),
            "target_continuation": target_clean,
            "generated_continuation": generated_continuation,
            "generation_max_new_tokens": generation_max_new_tokens,
            "generation_error": generation_error,
            "avg_nll": metrics["avg_nll"],
            "total_nll": metrics["total_nll"],
            "ppl": metrics["ppl"],
            "target_tokens": metrics["target_tokens"],
            "embedding_cosine_similarity": np.nan,
            "embedding_distance": np.nan,
            "error": error,
        }

        for col in METADATA_COLS:
            record[col] = row[col]

        records.append(record)

        example_time = time.time() - example_start_time
        average_time = (time.time() - start_time) / i

        progress_bar.set_postfix(
            step="done",
            last_s=round(example_time, 1),
            avg_s=round(average_time, 1),
            errors=n_errors,
            gen_errors=n_generation_errors,
            refresh=True,
        )

    results = pd.DataFrame(records)

    n_errors = int((results["error"] != "").sum()) if "error" in results.columns else n_errors
    n_generation_errors = (
        int((results["generation_error"] != "").sum())
        if "generation_error" in results.columns
        else n_generation_errors
    )

    print(
        f"{model_variant}: completed with {n_errors} NLL errors and "
        f"{n_generation_errors} generation errors out of {len(results)} examples."
    )

    return results


## 11. Evaluate the two non-fine-tuned baseline variants

This loads only the original pretrained Qwen3 model and tests it without metadata and with the full metadata block inserted in the prompt.

In [11]:
print("Loading base Qwen3 model...")
model, tokenizer = load_base_model_and_tokenizer()

base_outputs = {}

for variant_name, use_metadata in progress_tqdm(
    [
        ("base_no_metadata", False),
        ("base_with_metadata", True),
    ],
    desc="Base model evaluation",
    unit="variant",
    dynamic_ncols=True,
):
    base_outputs[variant_name] = evaluate_variant(
        model=model,
        tokenizer=tokenizer,
        dataframe=df_eval,
        model_variant=variant_name,
        use_metadata=use_metadata,
    )

base_no_metadata = base_outputs["base_no_metadata"]
base_with_metadata = base_outputs["base_with_metadata"]

# Free GPU memory before loading LoRA models.
del model, tokenizer
clear_memory()

base_results = pd.concat([base_no_metadata, base_with_metadata], ignore_index=True)
display(base_results.head())


Loading base Qwen3 model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Evaluating base_no_metadata:   0%|          | 0/300 [00:00<?, ?dream/s]

base_no_metadata: completed with 0 NLL errors and 0 generation errors out of 300 examples.


Evaluating base_with_metadata:   0%|          | 0/300 [00:00<?, ?dream/s]

base_with_metadata: completed with 0 NLL errors and 0 generation errors out of 300 examples.


,row_id,model_variant,use_metadata,dream_id,dreamer,metadata_text,n_words,prefix,target_continuation,generated_continuation,...,ppl,target_tokens,embedding_cosine_similarity,embedding_distance,error,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,0,base_no_metadata,False,0,,Dream metadata:\n- dominant_macro_emotion: joy...,191,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,and pulls it out quickly. The man has a knife ...,...,45.653921,72,NaN,NaN,,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown
1,1,base_no_metadata,False,1,,Dream metadata:\n- dominant_macro_emotion: cur...,53,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",Jeremy and I were sitting in a café together w...,...,24.198485,20,NaN,NaN,,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult
2,2,base_no_metadata,False,2,,Dream metadata:\n- dominant_macro_emotion: dis...,68,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,"The next day, when I went back to my office, t...",...,49.157494,25,NaN,NaN,,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group
3,3,base_no_metadata,False,3,,Dream metadata:\n- dominant_macro_emotion: joy...,70,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...",a bad dream about the death of my father and m...,...,25.520034,27,NaN,NaN,,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages
4,4,base_no_metadata,False,4,,Dream metadata:\n- dominant_macro_emotion: dis...,126,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,and it's just so hard to write with that thing...,...,16.865111,43,NaN,NaN,,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group


## 12. Evaluate fine-tuned Model A: no metadata

This loads the base model plus the `final_A_no_metadata` LoRA adapter produced by your fine-tuning notebook.

In [12]:
print("Loading fine-tuned LoRA Model A: no metadata...")
model, tokenizer = load_lora_model_and_tokenizer(ADAPTER_A_DIR)

finetuned_no_metadata = evaluate_variant(
    model=model,
    tokenizer=tokenizer,
    dataframe=df_eval,
    model_variant="finetuned_no_metadata",
    use_metadata=False,
)

del model, tokenizer
clear_memory()

display(finetuned_no_metadata.head())


Loading fine-tuned LoRA Model A: no metadata...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Evaluating finetuned_no_metadata:   0%|          | 0/300 [00:00<?, ?dream/s]

finetuned_no_metadata: completed with 0 NLL errors and 0 generation errors out of 300 examples.


,row_id,model_variant,use_metadata,dream_id,dreamer,metadata_text,n_words,prefix,target_continuation,generated_continuation,...,ppl,target_tokens,embedding_cosine_similarity,embedding_distance,error,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,0,finetuned_no_metadata,False,0,,Dream metadata:\n- dominant_macro_emotion: joy...,191,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,so I pull out my knife and stab him through hi...,...,34.667754,72,NaN,NaN,,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown
1,1,finetuned_no_metadata,False,1,,Dream metadata:\n- dominant_macro_emotion: cur...,53,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",I thought he had a cavity in his mouth but the...,...,8.711824,20,NaN,NaN,,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult
2,2,finetuned_no_metadata,False,2,,Dream metadata:\n- dominant_macro_emotion: dis...,68,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,"It did, and then we went out for a walk together.",...,29.019585,25,NaN,NaN,,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group
3,3,finetuned_no_metadata,False,3,,Dream metadata:\n- dominant_macro_emotion: joy...,70,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...","hurt."" He says, ""I'm not going to die!""",...,16.441314,27,NaN,NaN,,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages
4,4,finetuned_no_metadata,False,4,,Dream metadata:\n- dominant_macro_emotion: dis...,126,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,do with it so I got up and went out into the h...,...,9.210486,43,NaN,NaN,,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group


## 13. Evaluate fine-tuned Model B: with all metadata

This loads the base model plus the `final_B_with_metadata` LoRA adapter produced by your fine-tuning notebook.

In [13]:
print("Loading fine-tuned LoRA Model B: with metadata...")
model, tokenizer = load_lora_model_and_tokenizer(ADAPTER_B_DIR)

finetuned_with_metadata = evaluate_variant(
    model=model,
    tokenizer=tokenizer,
    dataframe=df_eval,
    model_variant="finetuned_with_metadata",
    use_metadata=True,
)

del model, tokenizer
clear_memory()

display(finetuned_with_metadata.head())


Loading fine-tuned LoRA Model B: with metadata...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Evaluating finetuned_with_metadata:   0%|          | 0/300 [00:00<?, ?dream/s]

finetuned_with_metadata: completed with 0 NLL errors and 0 generation errors out of 300 examples.


,row_id,model_variant,use_metadata,dream_id,dreamer,metadata_text,n_words,prefix,target_continuation,generated_continuation,...,ppl,target_tokens,embedding_cosine_similarity,embedding_distance,error,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,0,finetuned_with_metadata,True,0,,Dream metadata:\n- dominant_macro_emotion: joy...,191,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,then bites down hard. I try to pull away but c...,...,34.898180,72,NaN,NaN,,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown
1,1,finetuned_with_metadata,True,1,,Dream metadata:\n- dominant_macro_emotion: cur...,53,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",I thought he had a cavity in his mouth but the...,...,8.209667,20,NaN,NaN,,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult
2,2,finetuned_with_metadata,True,2,,Dream metadata:\n- dominant_macro_emotion: dis...,68,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,"It did, and then we went out for a walk together.",...,29.926677,25,NaN,NaN,,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group
3,3,finetuned_with_metadata,True,3,,Dream metadata:\n- dominant_macro_emotion: joy...,70,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...","hurt."" He says, ""Oh, no! No!""",...,15.856086,27,NaN,NaN,,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages
4,4,finetuned_with_metadata,True,4,,Dream metadata:\n- dominant_macro_emotion: dis...,126,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,do with it so I got up and went out into the h...,...,9.053071,43,NaN,NaN,,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group


## 14. Combine and save all four result tables

In [14]:
all_results = pd.concat(
    [
        base_no_metadata,
        base_with_metadata,
        finetuned_no_metadata,
        finetuned_with_metadata,
    ],
    ignore_index=True,
)

print("Combined all four result tables.")
print("Shape before embedding similarity:", all_results.shape)
display(all_results.head())


Combined all four result tables.
Shape before embedding similarity: (1200, 24)


,row_id,model_variant,use_metadata,dream_id,dreamer,metadata_text,n_words,prefix,target_continuation,generated_continuation,...,ppl,target_tokens,embedding_cosine_similarity,embedding_distance,error,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age
0,0,base_no_metadata,False,0,,Dream metadata:\n- dominant_macro_emotion: joy...,191,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,and pulls it out quickly. The man has a knife ...,...,45.653921,72,NaN,NaN,,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown
1,1,base_no_metadata,False,1,,Dream metadata:\n- dominant_macro_emotion: cur...,53,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",Jeremy and I were sitting in a café together w...,...,24.198485,20,NaN,NaN,,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult
2,2,base_no_metadata,False,2,,Dream metadata:\n- dominant_macro_emotion: dis...,68,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,"The next day, when I went back to my office, t...",...,49.157494,25,NaN,NaN,,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group
3,3,base_no_metadata,False,3,,Dream metadata:\n- dominant_macro_emotion: joy...,70,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...",a bad dream about the death of my father and m...,...,25.520034,27,NaN,NaN,,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages
4,4,base_no_metadata,False,4,,Dream metadata:\n- dominant_macro_emotion: dis...,126,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,and it's just so hard to write with that thing...,...,16.865111,43,NaN,NaN,,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group


## 15. Semantic embedding similarity between real and generated endings

This cell performs the extra semantic evaluation requested for the presentation.

For each generated continuation, it compares:

- the **true continuation** from the dataset
- the **generated continuation** produced by the model

The metric is **embedding cosine similarity**:

- values closer to **1** mean more similar meaning
- values closer to **0** mean weak semantic similarity
- this metric can capture paraphrases better than NLL/perplexity, because it is not based on exact token matching

This is the heavy computation, so it is done here. The VS Code interactive notebook will only load the exported CSVs and plot the results.


In [15]:
def load_sentence_embedding_model():
    if SentenceTransformer is None:
        raise ImportError(
            "sentence-transformers is not available. Run the install cell first, "
            "then restart the runtime if needed."
        )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} on {device}")
    return SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)


def add_embedding_similarity(results_df):
    results_df = results_df.copy()

    if not GENERATE_CONTINUATIONS_FOR_SEMANTIC_EVAL:
        print("Semantic generation was disabled, so embedding similarity is not computed.")
        return results_df

    valid_mask = (
        results_df["error"].eq("")
        & results_df["generation_error"].eq("")
        & results_df["target_continuation"].fillna("").astype(str).str.strip().ne("")
        & results_df["generated_continuation"].fillna("").astype(str).str.strip().ne("")
    )

    n_valid = int(valid_mask.sum())
    print(f"Rows available for embedding similarity: {n_valid} / {len(results_df)}")

    if n_valid == 0:
        print("No valid generated continuations found. Skipping embedding similarity.")
        return results_df

    embedder = load_sentence_embedding_model()

    target_texts = results_df.loc[valid_mask, "target_continuation"].astype(str).tolist()
    generated_texts = results_df.loc[valid_mask, "generated_continuation"].astype(str).tolist()

    target_embeddings = embedder.encode(
        target_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    generated_embeddings = embedder.encode(
        generated_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    cosine_similarities = np.sum(target_embeddings * generated_embeddings, axis=1)
    cosine_similarities = cosine_similarities.astype(float)

    results_df.loc[valid_mask, "embedding_cosine_similarity"] = cosine_similarities
    results_df.loc[valid_mask, "embedding_distance"] = 1.0 - cosine_similarities

    # Extra simple generation diagnostics.
    results_df["target_word_count"] = results_df["target_continuation"].fillna("").astype(str).str.split().str.len()
    results_df["generated_word_count"] = results_df["generated_continuation"].fillna("").astype(str).str.split().str.len()
    results_df["generated_to_target_word_ratio"] = (
        results_df["generated_word_count"] / results_df["target_word_count"].replace(0, np.nan)
    )

    # Release embedding model memory before the next cells.
    del embedder
    clear_memory()

    return results_df


all_results = add_embedding_similarity(all_results)

# Save detailed results only after semantic metrics have been added.
all_results.to_csv(RESULTS_CSV, index=False)
print("Saved detailed results with semantic metrics to:", RESULTS_CSV)
print("Shape after embedding similarity:", all_results.shape)

semantic_cols = [
    "row_id",
    "model_variant",
    "target_continuation",
    "generated_continuation",
    "embedding_cosine_similarity",
    "embedding_distance",
    "target_word_count",
    "generated_word_count",
    "generated_to_target_word_ratio",
    "generation_error",
]
available_semantic_cols = [col for col in semantic_cols if col in all_results.columns]
semantic_results = all_results[available_semantic_cols].copy()
semantic_results.to_csv(SEMANTIC_CSV, index=False)
print("Saved semantic similarity table to:", SEMANTIC_CSV)

display(all_results.head())


Rows available for embedding similarity: 1200 / 1200
Loading embedding model: BAAI/bge-small-en-v1.5 on cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Saved detailed results with semantic metrics to: qwen3_dream_four_model_metadata_evaluation_results.csv
Shape after embedding similarity: (1200, 27)
Saved semantic similarity table to: qwen3_dream_four_model_metadata_semantic_similarity.csv


,row_id,model_variant,use_metadata,dream_id,dreamer,metadata_text,n_words,prefix,target_continuation,generated_continuation,...,embedding_distance,error,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,target_word_count,generated_word_count,generated_to_target_word_ratio
0,0,base_no_metadata,False,0,,Dream metadata:\n- dominant_macro_emotion: joy...,191,Bigger Burger For $1.09 I'm just eaten some fo...,biting it. I swing my arm away and he is throw...,and pulls it out quickly. The man has a knife ...,...,0.528434,,joy_amusement,admiration_approval,topic_04_cookies_bread_grocery_store_pan,male,unknown,58,35,0.603448
1,1,base_no_metadata,False,1,,Dream metadata:\n- dominant_macro_emotion: cur...,53,I was walking around on campus. To my surprise...,"it was just one, but as I tried to put it back...",Jeremy and I were sitting in a café together w...,...,0.496950,,curiosity_surprise,excitement_pride,topic_23_tooth_ward_surgery_patients,female,young_adult,16,36,2.250000
2,2,base_no_metadata,False,2,,Dream metadata:\n- dominant_macro_emotion: dis...,68,I remember this dream very vaguely: The surrou...,The problems continued. Then I tried to get th...,"The next day, when I went back to my office, t...",...,0.272368,,disappointment_remorse,admiration_approval,topic_20_media_organ_celine_modern_media,female,mixed_or_group,21,35,1.666667
3,3,base_no_metadata,False,3,,Dream metadata:\n- dominant_macro_emotion: joy...,70,Father Andrew is on a trip: there is an Indian...,"his right hand between an arrow and a tree."" W...",a bad dream about the death of my father and m...,...,0.511564,,joy_amusement,excitement_pride,topic_21_mass_camden_prayer_pew,female,longitudinal_multiple_ages,21,36,1.714286
4,4,base_no_metadata,False,4,,Dream metadata:\n- dominant_macro_emotion: dis...,126,I'm trying to type on a little tiny portable t...,do about it and I was really frustrated and th...,and it's just so hard to write with that thing...,...,0.539178,,disappointment_remorse,fear_anxiety,topic_16_document_badge_agency_jennifer,female,mixed_or_group,38,44,1.157895


## 16. Summary metrics

The most important final metric is usually `weighted_avg_nll` and its corresponding `weighted_perplexity`.

In [16]:
def summarize_results(results_df):
    rows = []
    valid = results_df[results_df["error"].eq("")].copy()

    for variant, group in valid.groupby("model_variant"):
        total_nll = group["total_nll"].sum()
        total_tokens = group["target_tokens"].sum()
        weighted_avg_nll = total_nll / total_tokens

        row = {
            "model_variant": variant,
            "n_examples": int(len(group)),
            "total_target_tokens": int(total_tokens),
            "mean_avg_nll": float(group["avg_nll"].mean()),
            "median_avg_nll": float(group["avg_nll"].median()),
            "weighted_avg_nll": float(weighted_avg_nll),
            "mean_ppl": float(group["ppl"].mean()),
            "median_ppl": float(group["ppl"].median()),
            "weighted_perplexity": safe_exp(weighted_avg_nll),
        }

        if "embedding_cosine_similarity" in group.columns:
            semantic_group = group.dropna(subset=["embedding_cosine_similarity"]).copy()
            row["n_semantic_examples"] = int(len(semantic_group))

            if len(semantic_group) > 0:
                row["mean_embedding_similarity"] = float(semantic_group["embedding_cosine_similarity"].mean())
                row["median_embedding_similarity"] = float(semantic_group["embedding_cosine_similarity"].median())
                row["std_embedding_similarity"] = float(semantic_group["embedding_cosine_similarity"].std(ddof=0))
                row["mean_embedding_distance"] = float(semantic_group["embedding_distance"].mean())
                row["share_embedding_similarity_ge_0_50"] = float((semantic_group["embedding_cosine_similarity"] >= 0.50).mean())
                row["share_embedding_similarity_ge_0_70"] = float((semantic_group["embedding_cosine_similarity"] >= 0.70).mean())
                row["mean_generated_to_target_word_ratio"] = float(semantic_group["generated_to_target_word_ratio"].mean())
            else:
                row["mean_embedding_similarity"] = np.nan
                row["median_embedding_similarity"] = np.nan
                row["std_embedding_similarity"] = np.nan
                row["mean_embedding_distance"] = np.nan
                row["share_embedding_similarity_ge_0_50"] = np.nan
                row["share_embedding_similarity_ge_0_70"] = np.nan
                row["mean_generated_to_target_word_ratio"] = np.nan

        rows.append(row)

    summary = pd.DataFrame(rows).sort_values("weighted_avg_nll").reset_index(drop=True)
    return summary

summary = summarize_results(all_results)
summary.to_csv(SUMMARY_CSV, index=False)

print("Saved summary to:", SUMMARY_CSV)
display(summary)

best_nll = summary.iloc[0]
print(
    f"Best model by weighted average NLL: {best_nll['model_variant']} "
    f"with weighted_avg_nll={best_nll['weighted_avg_nll']:.4f} "
    f"and weighted_perplexity={best_nll['weighted_perplexity']:.4f}."
)

if "mean_embedding_similarity" in summary.columns and summary["mean_embedding_similarity"].notna().any():
    best_semantic = summary.sort_values("mean_embedding_similarity", ascending=False).iloc[0]
    print(
        f"Best model by mean embedding similarity: {best_semantic['model_variant']} "
        f"with mean_embedding_similarity={best_semantic['mean_embedding_similarity']:.4f}."
    )


Saved summary to: qwen3_dream_four_model_metadata_summary.csv


,model_variant,n_examples,total_target_tokens,mean_avg_nll,median_avg_nll,weighted_avg_nll,mean_ppl,median_ppl,weighted_perplexity,n_semantic_examples,mean_embedding_similarity,median_embedding_similarity,std_embedding_similarity,mean_embedding_distance,share_embedding_similarity_ge_0_50,share_embedding_similarity_ge_0_70,mean_generated_to_target_word_ratio
0,finetuned_with_metadata,300,16714,2.753428,2.717788,2.784761,17.677802,15.146778,16.195953,300,0.600394,0.599356,0.090246,0.399606,0.840000,0.143333,0.467192
1,finetuned_no_metadata,300,16714,2.755634,2.717669,2.785777,17.725330,15.144975,16.212405,300,0.599049,0.599640,0.089238,0.400951,0.860000,0.140000,0.481165
2,base_no_metadata,300,16714,3.183665,3.113346,3.145519,27.433567,22.496201,23.231729,300,0.629547,0.632125,0.087299,0.370453,0.936667,0.206667,1.320875
3,base_with_metadata,300,16714,3.205919,3.130174,3.159219,28.047437,22.878006,23.552183,300,0.630408,0.640753,0.092517,0.369592,0.903333,0.250000,1.307439


Best model by weighted average NLL: finetuned_with_metadata with weighted_avg_nll=2.7848 and weighted_perplexity=16.1960.
Best model by mean embedding similarity: base_with_metadata with mean_embedding_similarity=0.6304.


## 17. Pairwise comparisons

A positive `mean_nll_gain` means the second model in the comparison is better because it has lower NLL.

In [17]:
def build_pairwise_comparisons(results_df):
    valid = results_df[results_df["error"].eq("")].copy()

    nll_table = valid.pivot_table(
        index="row_id",
        columns="model_variant",
        values="avg_nll",
        aggfunc="first",
    )

    semantic_table = None
    if "embedding_cosine_similarity" in valid.columns:
        semantic_table = valid.pivot_table(
            index="row_id",
            columns="model_variant",
            values="embedding_cosine_similarity",
            aggfunc="first",
        )

    comparisons = [
        ("Metadata effect on base model", "base_no_metadata", "base_with_metadata"),
        ("Fine-tuning effect without metadata", "base_no_metadata", "finetuned_no_metadata"),
        ("Fine-tuning effect with metadata", "base_with_metadata", "finetuned_with_metadata"),
        ("Metadata + fine-tuning total effect", "base_no_metadata", "finetuned_with_metadata"),
        ("Metadata effect after fine-tuning", "finetuned_no_metadata", "finetuned_with_metadata"),
    ]

    rows = []
    for label, first, second in comparisons:
        if first not in nll_table.columns or second not in nll_table.columns:
            continue

        pair_nll = nll_table[[first, second]].dropna().copy()
        nll_gain = pair_nll[first] - pair_nll[second]  # positive means the second model has lower NLL

        row = {
            "comparison": label,
            "first_model": first,
            "second_model": second,
            "n_examples": int(len(pair_nll)),
            "mean_nll_first": float(pair_nll[first].mean()),
            "mean_nll_second": float(pair_nll[second].mean()),
            "mean_nll_gain": float(nll_gain.mean()),
            "median_nll_gain": float(nll_gain.median()),
            "share_examples_second_better": float((nll_gain > 0).mean()),
        }

        if semantic_table is not None and first in semantic_table.columns and second in semantic_table.columns:
            pair_semantic = semantic_table[[first, second]].dropna().copy()
            semantic_gain = pair_semantic[second] - pair_semantic[first]  # positive means the second model is semantically closer

            row.update({
                "n_semantic_examples": int(len(pair_semantic)),
                "mean_embedding_similarity_first": float(pair_semantic[first].mean()) if len(pair_semantic) else np.nan,
                "mean_embedding_similarity_second": float(pair_semantic[second].mean()) if len(pair_semantic) else np.nan,
                "mean_embedding_similarity_gain": float(semantic_gain.mean()) if len(pair_semantic) else np.nan,
                "median_embedding_similarity_gain": float(semantic_gain.median()) if len(pair_semantic) else np.nan,
                "share_examples_second_better_embedding": float((semantic_gain > 0).mean()) if len(pair_semantic) else np.nan,
            })

        rows.append(row)

    return pd.DataFrame(rows)

pairwise = build_pairwise_comparisons(all_results)
pairwise.to_csv(PAIRWISE_CSV, index=False)

print("Saved pairwise comparisons to:", PAIRWISE_CSV)
display(pairwise)


Saved pairwise comparisons to: qwen3_dream_four_model_metadata_pairwise_comparisons.csv


,comparison,first_model,second_model,n_examples,mean_nll_first,mean_nll_second,mean_nll_gain,median_nll_gain,share_examples_second_better,n_semantic_examples,mean_embedding_similarity_first,mean_embedding_similarity_second,mean_embedding_similarity_gain,median_embedding_similarity_gain,share_examples_second_better_embedding
0,Metadata effect on base model,base_no_metadata,base_with_metadata,300,3.183665,3.205919,-0.022253,-0.015970,0.316667,300,0.629547,0.630408,0.000862,0.000386,0.503333
1,Fine-tuning effect without metadata,base_no_metadata,finetuned_no_metadata,300,3.183665,2.755634,0.428032,0.366463,1.000000,300,0.629547,0.599049,-0.030498,-0.029640,0.360000
2,Fine-tuning effect with metadata,base_with_metadata,finetuned_with_metadata,300,3.205919,2.753428,0.452491,0.376421,1.000000,300,0.630408,0.600394,-0.030015,-0.029066,0.383333
3,Metadata + fine-tuning total effect,base_no_metadata,finetuned_with_metadata,300,3.183665,2.753428,0.430238,0.360527,1.000000,300,0.629547,0.600394,-0.029153,-0.025037,0.376667
4,Metadata effect after fine-tuning,finetuned_no_metadata,finetuned_with_metadata,300,2.755634,2.753428,0.002206,0.003958,0.560000,300,0.599049,0.600394,0.001345,0.000000,0.446667


## 18. No plots in this notebook

This notebook intentionally does **not** create plots.

The heavy evaluation work is completed here and exported to CSV files. The VS Code interactive notebook loads these CSV files and creates the presentation-ready graphs for:

- weighted average NLL
- weighted perplexity
- embedding cosine similarity
- pairwise improvements
- per-example distributions


In [18]:
print("No plots are generated in this notebook.")
print("Use the VS Code interactive notebook to visualize NLL, perplexity, and embedding similarity.")


No plots are generated in this notebook.
Use the VS Code interactive notebook to visualize NLL, perplexity, and embedding similarity.


## 19. Interpretation helper for your project report

Run this cell after the summary table. It prints a short interpretation you can adapt for the exam.

In [19]:
def format_signed(x):
    return f"{x:+.4f}" if pd.notna(x) else "nan"


print("FINAL INTERPRETATION")
print("=" * 80)
print("Lower NLL and lower perplexity mean that the model assigns higher probability to the true continuation of the dream.")
print("Higher embedding cosine similarity means that the generated continuation is semantically closer to the true continuation.")
print()

print("Ranking by weighted average NLL:")
for i, row in summary.sort_values("weighted_avg_nll").iterrows():
    print(
        f"- {row['model_variant']}: "
        f"weighted_avg_nll={row['weighted_avg_nll']:.4f}, "
        f"weighted_ppl={row['weighted_perplexity']:.4f}"
    )

if "mean_embedding_similarity" in summary.columns and summary["mean_embedding_similarity"].notna().any():
    print("\nRanking by mean embedding similarity:")
    for _, row in summary.sort_values("mean_embedding_similarity", ascending=False).iterrows():
        print(
            f"- {row['model_variant']}: "
            f"mean_embedding_similarity={row['mean_embedding_similarity']:.4f}, "
            f"median_embedding_similarity={row['median_embedding_similarity']:.4f}"
        )

print("\nPairwise interpretation:")
for _, row in pairwise.iterrows():
    nll_direction = "improves" if row["mean_nll_gain"] > 0 else "does not improve"
    line = (
        f"- {row['comparison']}: {nll_direction} average NLL "
        f"(mean NLL gain = {format_signed(row['mean_nll_gain'])}; "
        f"second model lower NLL in {row['share_examples_second_better']:.1%} of examples)"
    )

    if "mean_embedding_similarity_gain" in row and pd.notna(row["mean_embedding_similarity_gain"]):
        semantic_direction = "improves" if row["mean_embedding_similarity_gain"] > 0 else "does not improve"
        line += (
            f"; {semantic_direction} semantic similarity "
            f"(mean embedding gain = {format_signed(row['mean_embedding_similarity_gain'])}; "
            f"second model higher similarity in {row['share_examples_second_better_embedding']:.1%} of examples)"
        )

    print(line + ".")

print("\nImportant caveat:")
print(
    "NLL/perplexity are token-level metrics: they reward assigning probability to the exact real wording. "
    "Embedding similarity is more semantic, but it is still automatic and should be complemented with qualitative examples."
)
print(
    "If the same CSV was used for final fine-tuning and evaluation, the fine-tuned results are optimistic. "
    "For a true generalization estimate, evaluate on a held-out test set or rely on cross-validation metrics."
)


FINAL INTERPRETATION
Lower NLL and lower perplexity mean that the model assigns higher probability to the true continuation of the dream.
Higher embedding cosine similarity means that the generated continuation is semantically closer to the true continuation.

Ranking by weighted average NLL:
- finetuned_with_metadata: weighted_avg_nll=2.7848, weighted_ppl=16.1960
- finetuned_no_metadata: weighted_avg_nll=2.7858, weighted_ppl=16.2124
- base_no_metadata: weighted_avg_nll=3.1455, weighted_ppl=23.2317
- base_with_metadata: weighted_avg_nll=3.1592, weighted_ppl=23.5522

Ranking by mean embedding similarity:
- base_with_metadata: mean_embedding_similarity=0.6304, median_embedding_similarity=0.6408
- base_no_metadata: mean_embedding_similarity=0.6295, median_embedding_similarity=0.6321
- finetuned_with_metadata: mean_embedding_similarity=0.6004, median_embedding_similarity=0.5994
- finetuned_no_metadata: mean_embedding_similarity=0.5990, median_embedding_similarity=0.5996

Pairwise interpret

## 20. Optional: inspect worst and best examples

This helps you understand where the metadata block helps or hurts.

In [20]:
# Example: compare base without metadata vs base with metadata.
valid = all_results[all_results["error"].eq("")].copy()
wide_nll = valid.pivot_table(index="row_id", columns="model_variant", values="avg_nll", aggfunc="first")

if {"base_no_metadata", "base_with_metadata"}.issubset(set(wide_nll.columns)):
    wide_nll["metadata_gain_base_nll"] = wide_nll["base_no_metadata"] - wide_nll["base_with_metadata"]
    examples = df_eval.merge(wide_nll[["metadata_gain_base_nll"]].reset_index(), on="row_id", how="left")

    if "embedding_cosine_similarity" in valid.columns:
        wide_semantic = valid.pivot_table(
            index="row_id",
            columns="model_variant",
            values="embedding_cosine_similarity",
            aggfunc="first",
        )
        if {"base_no_metadata", "base_with_metadata"}.issubset(set(wide_semantic.columns)):
            wide_semantic["metadata_gain_base_embedding"] = (
                wide_semantic["base_with_metadata"] - wide_semantic["base_no_metadata"]
            )
            examples = examples.merge(
                wide_semantic[["metadata_gain_base_embedding"]].reset_index(),
                on="row_id",
                how="left",
            )

    display_cols = ["row_id"] + METADATA_COLS + ["n_words", "metadata_gain_base_nll"]
    if "metadata_gain_base_embedding" in examples.columns:
        display_cols.append("metadata_gain_base_embedding")
    display_cols += ["prefix", "continuation"]

    print("Examples where metadata helps the base model most by NLL:")
    display(examples.sort_values("metadata_gain_base_nll", ascending=False)[display_cols].head(5))

    print("Examples where metadata hurts the base model most by NLL:")
    display(examples.sort_values("metadata_gain_base_nll", ascending=True)[display_cols].head(5))

    if "metadata_gain_base_embedding" in examples.columns:
        print("Examples where metadata helps the base model most by embedding similarity:")
        display(examples.sort_values("metadata_gain_base_embedding", ascending=False)[display_cols].head(5))
else:
    print("Required model variants not found in results.")


Examples where metadata helps the base model most by NLL:


,row_id,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words,metadata_gain_base_nll,metadata_gain_base_embedding,prefix,continuation
101,101,admiration_approval,excitement_pride,topic_12_canoe_shore_ship_swimming_pool,female,middle_aged,69,0.215142,-0.102017,I am an incredible swimmer. I dive off a divin...,and easily step out onto the pool's edge. I am...
124,124,embarrassment_confusion,curiosity_surprise,topic_00_passenger_highway_drivers_driving_car,female,longitudinal_multiple_ages,118,0.149138,0.153852,I was at university. There was a big lecture i...,there either. I went to the last one and found...
97,97,admiration_approval,anger_frustration,topic_02_wheelchair_bonnie_yarn_dovre,female,longitudinal_multiple_ages,52,0.132864,-0.035212,Waiting to be picked up after a swim. Jo. Not ...,there. First I say yes then no. I think I can ...
280,280,sadness_grief,anger_frustration,topic_14_dating_kissing_bj_nude,female,longitudinal_multiple_ages,82,0.112446,-0.010544,A wife found out her husband was cheating on h...,her. This new girl came into his life somehow ...
8,8,anger_frustration,disgust_disapproval,topic_03_aliens_enemy_guns_shooting,female,longitudinal_multiple_ages,58,0.109453,-0.028779,There was a girl or maybe two (I think it was ...,there and I was there as well and he started k...


Examples where metadata hurts the base model most by NLL:


,row_id,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words,metadata_gain_base_nll,metadata_gain_base_embedding,prefix,continuation
204,204,admiration_approval,love_caring,topic_06_deer_snake_hunting_bear,male,unknown,51,-0.237715,0.029126,"I was on my way somewhere, riding in an army t...",under its tail) against mine. The streets outs...
187,187,joy_amusement,admiration_approval,topic_01_ms_exam_math_elijah,female,longitudinal_multiple_ages,56,-0.236382,0.010039,Mom and Ezra were watching Freaks and Geeks on...,shoebox these days. Mom was doing something el...
223,223,disappointment_remorse,sadness_grief,topic_04_cookies_bread_grocery_store_pan,female,unknown,57,-0.205190,-0.094002,I was with my husband. We were in the car and ...,reason and I was sort of irritated by this bec...
183,183,joy_amusement,excitement_pride,topic_24_ceremony_bride_marrying_groom,mixed_or_group,mixed_or_group,74,-0.183814,0.004251,I am getting married to this person who I can ...,kind of fuzzy on the sides and there is white ...
16,16,admiration_approval,anger_frustration,topic_11_pee_diaper_tub_bath,male,unknown,69,-0.152524,-0.068598,Drunk Nate I go into my living room at home. M...,"going to sleep at my house, on the floor. I br..."


Examples where metadata helps the base model most by embedding similarity:


,row_id,dominant_macro_emotion,second_dominant_macro_emotion,dream_topic,sex,age,n_words,metadata_gain_base_nll,metadata_gain_base_embedding,prefix,continuation
170,170,admiration_approval,optimism_desire,topic_21_mass_camden_prayer_pew,female,mixed_or_group,107,-0.117233,0.223610,"I am walking over some difficult terrain, up a...","smooth, unlined, clean shaven, like a baby's c..."
193,193,optimism_desire,anger_frustration,topic_08_make_love_penis_sexually_annie,female,middle_aged,100,-0.041609,0.174636,"(09/03/97)[""Discrimination and Talk Show Hosts...",French Women. Rosie is surprised and feels bad...
79,79,anger_frustration,disappointment_remorse,topic_05_nana_nanas_poppa_darren,female,longitudinal_multiple_ages,85,-0.001080,0.168200,I was at home and it was really late. We got M...,girlfriend. Ezra was there and we were all goi...
212,212,admiration_approval,anger_frustration,topic_06_deer_snake_hunting_bear,female,longitudinal_multiple_ages,139,-0.035729,0.163100,I had one day of school left. That morning tho...,then the crocodile left. The camera shifted to...
284,284,curiosity_surprise,embarrassment_confusion,topic_12_canoe_shore_ship_swimming_pool,female,middle_aged,162,-0.009734,0.156796,Kenneth is leaning out of a window and turned ...,channel deep enough in it or we wouldn't be ab...


## 21. Optional: zip outputs for download

In [21]:
output_files = [RESULTS_CSV, SUMMARY_CSV, PAIRWISE_CSV, SEMANTIC_CSV]
existing_files = [f for f in output_files if Path(f).exists()]

if existing_files:
    zip_name = "qwen3_dream_evaluation_outputs"
    temp_dir = Path(zip_name)
    temp_dir.mkdir(exist_ok=True)

    for f in existing_files:
        shutil.copy(f, temp_dir / f)

    zip_path = shutil.make_archive(zip_name, "zip", str(temp_dir))
    print("Created:", zip_path)
    print("Files included:")
    for f in existing_files:
        print("-", f)

    if IN_COLAB:
        files.download(zip_path)
else:
    print("No output files found yet. Run the evaluation cells first.")


Created: /content/qwen3_dream_evaluation_outputs.zip
Files included:
- qwen3_dream_four_model_metadata_evaluation_results.csv
- qwen3_dream_four_model_metadata_summary.csv
- qwen3_dream_four_model_metadata_pairwise_comparisons.csv
- qwen3_dream_four_model_metadata_semantic_similarity.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>